# 05 — Pooled Stock-Level Machine Learning and Sector Aggregation

This notebook is the second ML branch of the study.

It trains pooled models on stock-date observations from the **same dynamic top-50 Materials portfolio** and then converts individual volatility forecasts into a sector-volatility forecast.

In [1]:
from pathlib import Path
import time, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

ROOT=Path.cwd().resolve()
if ROOT.name.lower()=="notebooks": ROOT=ROOT.parent
PROCESSED=ROOT/"data"/"processed"; TABLES=ROOT/"results"/"tables"; DIAG=ROOT/"results"/"diagnostics"
for p in [TABLES,DIAG]: p.mkdir(parents=True,exist_ok=True)
HORIZONS=[5,21,63]; TEST_START=pd.Timestamp("2025-01-01")
RS=42; EPS=1e-10; REFIT_EVERY=21; CORR_WINDOW=63

def qlike(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    r=np.maximum(y*y,EPS)/np.maximum(p*p,EPS)
    return float(np.mean(r-np.log(r)-1))

def models():
    out={}
    for a in [.1,1,10,100]:
        out[f"Ridge_a{a}"]=Pipeline([("imp",SimpleImputer(strategy="median")),
            ("sc",StandardScaler()),("m",Ridge(alpha=a))])
    for leaf in [5,15,30]:
        out[f"RF_leaf{leaf}"]=Pipeline([("imp",SimpleImputer(strategy="median")),
            ("m",RandomForestRegressor(n_estimators=300,min_samples_leaf=leaf,max_features=.7,n_jobs=-1,random_state=RS))])
    for leaf in [10,30,60]:
        out[f"HGB_leaf{leaf}"]=Pipeline([("imp",SimpleImputer(strategy="median")),
            ("m",HistGradientBoostingRegressor(max_iter=200,learning_rate=.05,min_samples_leaf=leaf,random_state=RS))])
    try:
        from xgboost import XGBRegressor
        for dep in [2,3]:
            out[f"XGB_d{dep}"]=Pipeline([("imp",SimpleImputer(strategy="median")),
                ("m",XGBRegressor(n_estimators=350,max_depth=dep,learning_rate=.03,subsample=.8,
                     colsample_bytree=.8,n_jobs=-1,random_state=RS,objective="reg:squarederror"))])
    except Exception:
        print("XGBoost unavailable; continuing with sklearn models.")
    return out

def load_stock(h):
    d=pd.read_parquet(PROCESSED/f"stock_level_features_h{h}.parquet").copy()
    d["Date"]=pd.to_datetime(d["Date"])
    d=d.sort_values(["Date","Yahoo_Ticker"]).reset_index(drop=True)
    y=f"Target_Stock_RV_{h}"
    exclude={"Date","Yahoo_Ticker","Stock_Return",
             "Target_Stock_RV_5","Target_Stock_RV_21","Target_Stock_RV_63"}
    features=[c for c in d.select_dtypes(include=[np.number]).columns if c not in exclude]
    return d,features,y

## 1. Pre-test model selection on 2023–2024 validation folds

In [2]:
rows=[]
for h in HORIZONS:
    d,features,y=load_stock(h)
    dates=pd.Index(sorted(d.Date.unique()))
    for yr in [2023,2024]:
        v0=pd.Timestamp(f"{yr}-01-01"); v1=pd.Timestamp(f"{yr}-12-31")
        first=np.searchsorted(dates.values,np.datetime64(v0))
        if first<h: continue
        cutoff=pd.Timestamp(dates[first-h])
        tr=d[(d.Date<=cutoff)&d[y].notna()]
        va=d[d.Date.between(v0,v1)&d[y].notna()]
        for name,m in models().items():
            for transform in ["raw","log"]:
                yy=tr[y].to_numpy(float)
                if transform=="log": yy=np.log(np.maximum(yy,1e-8))
                m.fit(tr[features],yy)
                p=np.asarray(m.predict(va[features]),float)
                if transform=="log": p=np.exp(p)
                p=np.maximum(p,1e-8)
                e=p-va[y].to_numpy(float)
                rows.append({"Horizon":h,"Validation_Year":yr,"Model":name,"Transform":transform,
                             "N_Train":len(tr),"N_Val":len(va),"QLIKE":qlike(va[y],p),
                             "MAE":np.mean(abs(e)),"RMSE":np.sqrt(np.mean(e*e)),
                             "Correlation":np.corrcoef(va[y],p)[0,1] if np.std(p)>0 else np.nan})
validation=pd.DataFrame(rows)
validation.to_csv(TABLES/"05_stock_level_validation_results.csv",index=False)

summary=(validation.groupby(["Horizon","Model","Transform"],as_index=False)
         .agg(Mean_Val_QLIKE=("QLIKE","mean"),SD_Val_QLIKE=("QLIKE","std"),
              Mean_Val_MAE=("MAE","mean"),Folds=("Validation_Year","nunique")))
selected=(summary[summary.Folds.eq(2)].sort_values(["Horizon","Mean_Val_QLIKE"])
          .groupby("Horizon",as_index=False).first())
selected.to_csv(TABLES/"05_selected_stock_level_model.csv",index=False)
display(selected)

,Horizon,Model,Transform,Mean_Val_QLIKE,SD_Val_QLIKE,Mean_Val_MAE,Folds
0,5,RF_leaf5,raw,1.265500,0.601886,0.198529,2
1,21,RF_leaf15,raw,0.739144,0.544231,0.147583,2
2,63,RF_leaf5,raw,0.522037,0.511512,0.130454,2


## 2. Expanding 2025+ stock forecasts and sector aggregation

In [3]:
def make_selected(name):
    return models()[name]

def run_horizon(h):
    d,features,y=load_stock(h)
    spec=selected[selected.Horizon.eq(h)].iloc[0]
    dates=pd.Index(sorted(d.Date.unique()))
    test_dates=[pd.Timestamp(x) for x in dates if pd.Timestamp(x)>=TEST_START
                and d.loc[d.Date.eq(pd.Timestamp(x)),y].notna().any()]
    retmat=d.pivot_table(index="Date",columns="Yahoo_Ticker",values="Stock_Return",aggfunc="first").sort_index()
    model=None; last_refit=None; stock_out=[]; sector_out=[]; audit=[]

    print(f"H{h}: {spec.Model}/{spec.Transform}; {len(test_dates)} origins")
    for j,dt in enumerate(test_dates):
        if model is None or j%REFIT_EVERY==0:
            pos=dates.get_loc(dt)
            cutoff=pd.Timestamp(dates[pos-h])
            tr=d[(d.Date<=cutoff)&d[y].notna()]
            model=make_selected(str(spec.Model))
            yy=tr[y].to_numpy(float)
            if spec.Transform=="log": yy=np.log(np.maximum(yy,1e-8))
            tic=time.time(); model.fit(tr[features],yy); secs=time.time()-tic
            last_refit=dt
            audit.append({"Horizon":h,"Forecast_Origin":dt,"Latest_Training_Label_Date":cutoff,
                          "N_Train":len(tr),"Model":spec.Model,"Transform":spec.Transform,"Fit_Seconds":secs})
            print(f"  refit {dt.date()} | {len(tr):,} rows | {secs:.1f}s")

        te=d[(d.Date==dt)&d[y].notna()].copy()
        if len(te)<2: continue
        p=np.asarray(model.predict(te[features]),float)
        if spec.Transform=="log": p=np.exp(p)
        te["Pred_Stock_RV"]=np.maximum(p,1e-8)
        te=te[np.isfinite(te.Weight)&(te.Weight>=0)&np.isfinite(te.Pred_Stock_RV)].copy()
        if len(te)<2 or te.Weight.sum()<=0: continue
        te["Weight"]=te.Weight/te.Weight.sum()

        tick=te.Yahoo_Ticker.tolist()
        hist=retmat.loc[retmat.index<dt,tick].tail(CORR_WINDOW)
        R=hist.corr(min_periods=20).reindex(index=tick,columns=tick).fillna(0.)
        np.fill_diagonal(R.values,1.)
        vals,vecs=np.linalg.eigh(R.to_numpy()); vals=np.clip(vals,1e-8,None)
        Rp=(vecs*vals)@vecs.T; scale=np.sqrt(np.diag(Rp)); Rp=Rp/np.outer(scale,scale)

        a=te.Weight.to_numpy(float)*te.Pred_Stock_RV.to_numpy(float)
        sector_pred=float(np.sqrt(max(a@Rp@a,EPS)))

        sector_out.append({"Date":dt,"Horizon":h,"Stock_Level_ML_Aggregated":sector_pred,
                           "N_Stocks":len(te),"Refit_Every":REFIT_EVERY,
                           "Correlation_Window":CORR_WINDOW,"Model":spec.Model,"Transform":spec.Transform})
        stock_out.append(te[["Date","Yahoo_Ticker","Weight",y,"Pred_Stock_RV"]].assign(Horizon=h))

    return (pd.concat(stock_out,ignore_index=True),pd.DataFrame(sector_out),pd.DataFrame(audit))

stocks=[]; sectors=[]; audits=[]
for h in HORIZONS:
    a,b,c=run_horizon(h); stocks.append(a); sectors.append(b); audits.append(c)
stock_pred=pd.concat(stocks,ignore_index=True)
sector_pred=pd.concat(sectors,ignore_index=True)
refit_audit=pd.concat(audits,ignore_index=True)

stock_pred.to_parquet(TABLES/"05_stock_level_predictions.parquet",index=False)
sector_pred.to_csv(TABLES/"05_stock_level_aggregated_sector_predictions.csv",index=False)
refit_audit.to_csv(DIAG/"05_stock_level_refit_audit.csv",index=False)
display(refit_audit.groupby("Horizon").agg(Refits=("Forecast_Origin","count"),
    Total_Fit_Minutes=("Fit_Seconds",lambda x:x.sum()/60)))

H5: RF_leaf5/raw; 430 origins
  refit 2025-01-02 | 88,014 rows | 61.9s
  refit 2025-02-03 | 89,060 rows | 58.0s
  refit 2025-03-04 | 90,105 rows | 60.1s
  refit 2025-04-02 | 91,150 rows | 66.0s
  refit 2025-05-06 | 92,198 rows | 63.4s
  refit 2025-06-04 | 93,240 rows | 63.2s
  refit 2025-07-04 | 94,290 rows | 63.4s
  refit 2025-08-04 | 95,335 rows | 66.8s
  refit 2025-09-02 | 96,380 rows | 69.7s
  refit 2025-10-01 | 97,425 rows | 71.2s
  refit 2025-10-30 | 98,475 rows | 69.9s
  refit 2025-11-28 | 99,521 rows | 76.3s
  refit 2025-12-31 | 100,558 rows | 66.7s
  refit 2026-02-02 | 101,602 rows | 75.2s
  refit 2026-03-03 | 102,648 rows | 70.5s
  refit 2026-04-01 | 103,689 rows | 76.1s
  refit 2026-05-04 | 104,726 rows | 75.3s
  refit 2026-06-02 | 105,765 rows | 71.0s
  refit 2026-07-02 | 106,810 rows | 70.4s
  refit 2026-07-31 | 107,851 rows | 80.2s
  refit 2026-08-31 | 108,882 rows | 80.4s
H21: RF_leaf15/raw; 414 origins
  refit 2025-01-02 | 86,086 rows | 43.3s
  refit 2025-02-03 | 87,127

,Refits,Total_Fit_Minutes
Horizon,,
5,21,24.259367
21,20,16.631443
63,18,14.861005


## 3. Stock-level predictive diagnostics

In [4]:
stock_metrics = []

for h in HORIZONS:

    y = f"Target_Stock_RV_{h}"
    z = stock_pred.loc[
        stock_pred["Horizon"].eq(h),
        [
            "Date",
            "Yahoo_Ticker",
            y,
            "Pred_Stock_RV"
        ]
    ].copy()

    z = z.replace(
        [np.inf, -np.inf],
        np.nan
    )

    z = z.dropna(
        subset=[y, "Pred_Stock_RV"]
    )

    if z.empty:
        raise ValueError(
            f"No valid stock-level test observations "
            f"remain for horizon {h}."
        )

    yy = z[y].to_numpy(dtype=float)
    pp = z["Pred_Stock_RV"].to_numpy(dtype=float)

    e = pp - yy

    if (
        len(z) > 1
        and np.std(yy) > 0
        and np.std(pp) > 0
    ):
        corr = np.corrcoef(yy, pp)[0, 1]
    else:
        corr = np.nan

    actual_std = np.std(yy, ddof=1)
    prediction_std = np.std(pp, ddof=1)

    stock_metrics.append({
        "Horizon": h,
        "N": len(z),
        "QLIKE": qlike(yy,pp),
        "MAE": np.mean(np.abs(e)),
        "RMSE": np.sqrt(np.mean(e ** 2)),
        "Correlation": corr,
        "Bias": np.mean(e),
        "Actual_Std": actual_std,
        "Prediction_Std": prediction_std,
        "Variability_Ratio":
            prediction_std / actual_std
            if actual_std > 0
            else np.nan
    })
stock_metrics = pd.DataFrame(
    stock_metrics
)
stock_metrics.to_csv(
    TABLES / "05_stock_level_test_metrics.csv",
    index=False
)
display(stock_metrics)

,Horizon,N,QLIKE,MAE,RMSE,Correlation,Bias,Actual_Std,Prediction_Std,Variability_Ratio
0,5,21361,2.468688,0.188058,0.283660,0.513336,0.019873,0.325077,0.214236,0.659032
1,21,20213,0.396688,0.136340,0.202333,0.628254,0.003837,0.253654,0.203869,0.803729
2,63,17446,0.213253,0.111220,0.156517,0.695076,-0.016123,0.212013,0.179083,0.844678
